# 4. CatBoost — Irrigation Need (Multiclass)

CatBoost maneja las **8 variables categóricas nativamente** (sin OrdinalEncoder),  
usando *ordered target statistics* que preservan la relación real con el target.

1. Preprocesamiento mínimo — categóricas como strings, numéricas sin escalar
2. Modelo base con Cross-Validation 5-fold (Accuracy + Log-Loss)
3. Tuning ligero con RandomizedSearchCV
4. Feature importance
5. Predicciones + submission

**Contexto:** XGBoost base obtuvo 0.97 public score. CatBoost apunta a igualar o superar ese resultado aprovechando el tratamiento nativo de categóricas.

**Dataset:** Playground Series S6E4 | **Clases:** Low / Medium / High | **Métrica:** Accuracy

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, log_loss
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

print("Librerías cargadas")

Librerías cargadas


## 1. Carga y Preprocesamiento

A diferencia de XGBoost, **no se aplica OrdinalEncoder** — CatBoost recibe las categóricas como strings  
y aprende sus estadísticos internamente.

In [2]:
train_raw  = pd.read_csv("../data/train.csv")
test_raw   = pd.read_csv("../data/test.csv")

print(f"Train:  {train_raw.shape[0]:,} filas x {train_raw.shape[1]} columnas")
print(f"Test:   {test_raw.shape[0]:,} filas x {test_raw.shape[1]} columnas")

Train:  630,000 filas x 21 columnas
Test:   270,000 filas x 20 columnas


In [3]:
TARGET      = "Irrigation_Need"
CLASS_ORDER = ["Low", "Medium", "High"]
LABEL_MAP   = {"Low": 0, "Medium": 1, "High": 2}
INV_MAP     = {0: "Low", 1: "Medium", 2: "High"}

continuas = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]
categoricas = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"
]
features = continuas + categoricas

# Índices de columnas categóricas (requerido por CatBoost)
cat_feature_indices = [features.index(c) for c in categoricas]

ids_test = test_raw["id"].values
y = train_raw[TARGET].map(LABEL_MAP).values

# Categóricas como string para CatBoost; numéricas sin transformar
X_train = train_raw[features].copy()
X_test  = test_raw[features].copy()
for col in categoricas:
    X_train[col] = X_train[col].astype(str)
    X_test[col]  = X_test[col].astype(str)

# Pesos de muestra: inverso de frecuencia de clase
class_counts   = np.bincount(y)
sample_weights = np.array([1.0 / class_counts[yi] for yi in y])
sample_weights /= sample_weights.mean()

print(f"{'='*65}")
print("PREPROCESAMIENTO COMPLETADO")
print(f"{'='*65}")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Índices de categóricas: {cat_feature_indices}")
print(f"\nDistribución de clases:")
for cls, idx in LABEL_MAP.items():
    pct = class_counts[idx] / len(y) * 100
    print(f"  {cls:8s} ({idx}): {class_counts[idx]:>7,}  ({pct:.2f}%)")
print(f"{'='*65}")

PREPROCESAMIENTO COMPLETADO
X_train: (630000, 19)
X_test:  (270000, 19)
Índices de categóricas: [11, 12, 13, 14, 15, 16, 17, 18]

Distribución de clases:
  Low      (0): 369,917  (58.72%)
  Medium   (1): 239,074  (37.95%)
  High     (2):  21,009  (3.33%)


## 2. Modelo Base — Cross-Validation 5-fold

In [4]:
base_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=False
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_accs   = []
base_losses = []
last_val_idx   = None
last_val_preds = None
last_val_y     = None

print(f"{'='*65}")
print("CROSS-VALIDATION — CatBoost Base (5-fold)")
print(f"{'='*65}")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
    X_tr  = X_train.iloc[tr_idx]
    X_val = X_train.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr = sample_weights[tr_idx]

    pool_tr  = Pool(X_tr,  y_tr,  cat_features=cat_feature_indices, weight=sw_tr)
    pool_val = Pool(X_val, y_val, cat_features=cat_feature_indices)

    base_model.fit(pool_tr, eval_set=pool_val, use_best_model=False, verbose=False)

    proba = base_model.predict_proba(pool_val)
    preds = np.argmax(proba, axis=1)

    acc = accuracy_score(y_val, preds)
    ll  = log_loss(y_val, proba)
    base_accs.append(acc)
    base_losses.append(ll)

    last_val_idx   = val_idx
    last_val_preds = preds
    last_val_y     = y_val

    print(f"  Fold {fold}: Accuracy = {acc:.4f}  |  Log-Loss = {ll:.4f}")

print(f"{'='*65}")
print(f"  Accuracy: {np.mean(base_accs):.4f} ± {np.std(base_accs):.4f}")
print(f"  Log-Loss: {np.mean(base_losses):.4f} ± {np.std(base_losses):.4f}")
print(f"{'='*65}")

CROSS-VALIDATION — CatBoost Base (5-fold)
  Fold 1: Accuracy = 0.9818  |  Log-Loss = 0.0693


KeyboardInterrupt: 

## 3. Tuning — RandomizedSearchCV

Buscamos sobre los hiperparámetros más influyentes de CatBoost.

In [ ]:
from sklearn.model_selection import train_test_split
from scipy.stats import randint, uniform

# Tunear sobre 20% estratificado para reducir tiempo
X_tune_df, _, y_tune, _ = train_test_split(
    X_train, y, test_size=0.80, stratify=y, random_state=42
)
sw_tune = sample_weights[X_tune_df.index]
print(f"Subsample para tuning: {X_tune_df.shape[0]:,} filas")

param_distributions = {
    "depth":           randint(4, 10),
    "learning_rate":   uniform(0.03, 0.27),
    "iterations":      randint(300, 1001),
    "l2_leaf_reg":     uniform(1, 9),
    "bagging_temperature": uniform(0, 1),
}

search_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="Accuracy",
    cat_features=cat_feature_indices,
    random_seed=42,
    verbose=False
)

random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="accuracy",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_tune_df, y_tune, sample_weight=sw_tune)

best_params   = random_search.best_params_
best_cv_score = random_search.best_score_

print(f"\n{'='*65}")
print("MEJORES HIPERPARÁMETROS (RandomizedSearchCV)")
print(f"{'='*65}")
for param, val in best_params.items():
    print(f"  {param:25s}: {val}")
print(f"\n  CV Accuracy (mejor, subsample): {best_cv_score:.4f}")
print(f"  CatBoost Base CV Acc:           {np.mean(base_accs):.4f}")
print(f"  Diferencia:                     {best_cv_score - np.mean(base_accs):+.4f}")
print(f"{'='*65}")

## 4. Modelo Tuned — Validación CV

In [ ]:
tuned_model = CatBoostClassifier(
    **best_params,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=False
)

tuned_accs   = []
tuned_losses = []

print(f"{'='*65}")
print("CROSS-VALIDATION — CatBoost Tuned (5-fold)")
print(f"{'='*65}")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
    X_tr  = X_train.iloc[tr_idx]
    X_val = X_train.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr = sample_weights[tr_idx]

    pool_tr  = Pool(X_tr,  y_tr,  cat_features=cat_feature_indices, weight=sw_tr)
    pool_val = Pool(X_val, y_val, cat_features=cat_feature_indices)

    tuned_model.fit(pool_tr, eval_set=pool_val, use_best_model=False, verbose=False)

    proba = tuned_model.predict_proba(pool_val)
    preds = np.argmax(proba, axis=1)

    acc = accuracy_score(y_val, preds)
    ll  = log_loss(y_val, proba)
    tuned_accs.append(acc)
    tuned_losses.append(ll)

    print(f"  Fold {fold}: Accuracy = {acc:.4f}  |  Log-Loss = {ll:.4f}")

print(f"{'='*65}")
print(f"  Tuned — Accuracy: {np.mean(tuned_accs):.4f} ± {np.std(tuned_accs):.4f}")
print(f"  Base  — Accuracy: {np.mean(base_accs):.4f} ± {np.std(base_accs):.4f}")
print(f"  Mejora:           {np.mean(tuned_accs) - np.mean(base_accs):+.4f}")
print(f"{'='*65}")

# Seleccionar el mejor entre base y tuned
if np.mean(tuned_accs) >= np.mean(base_accs):
    best_model = tuned_model
    best_accs  = tuned_accs
    best_label = "Tuned"
else:
    best_model = base_model
    best_accs  = base_accs
    best_label = "Base"
print(f"\n  >> Modelo seleccionado: CatBoost {best_label} ({np.mean(best_accs):.4f})")

In [ ]:
folds = np.arange(1, 6)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(folds, base_accs,  "o-", color="steelblue", label=f"Base  (media={np.mean(base_accs):.4f})")
ax.plot(folds, tuned_accs, "s-", color="tomato",    label=f"Tuned (media={np.mean(tuned_accs):.4f})")
ax.set_title("Accuracy por fold", fontsize=13, fontweight="bold")
ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_xticks(folds)

ax = axes[1]
ax.plot(folds, base_losses,  "o-", color="steelblue", label=f"Base  (media={np.mean(base_losses):.4f})")
ax.plot(folds, tuned_losses, "s-", color="tomato",    label=f"Tuned (media={np.mean(tuned_losses):.4f})")
ax.set_title("Log-Loss por fold", fontsize=13, fontweight="bold")
ax.set_xlabel("Fold"); ax.set_ylabel("Log-Loss")
ax.legend(); ax.set_xticks(folds)

plt.suptitle("Comparativa: CatBoost Base vs Tuned", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# Entrenar el mejor modelo sobre todo el train
pool_full = Pool(X_train, y, cat_features=cat_feature_indices, weight=sample_weights)
best_model.fit(pool_full, verbose=False)

importances = best_model.get_feature_importance(pool_full)
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

print(f"{'='*65}")
print(f"FEATURE IMPORTANCE — CatBoost {best_label}")
print(f"{'='*65}")
for feat, imp in feat_imp.sort_values(ascending=False).items():
    bar = "█" * int(imp / 2)
    print(f"  {feat:35s}: {imp:6.2f}  {bar}")
print(f"{'='*65}")

colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp)))
plt.figure(figsize=(12, max(5, len(feat_imp) * 0.45)))
plt.barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor="black")
plt.xlabel("Importancia", fontsize=12, fontweight="bold")
plt.title(f"Feature Importance — CatBoost {best_label}", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 6. Predicciones y Submission

In [ ]:
pool_test  = Pool(X_test, cat_features=cat_feature_indices)
test_proba = best_model.predict_proba(pool_test)
test_preds = np.argmax(test_proba, axis=1)
test_labels = [INV_MAP[i] for i in test_preds]

print(f"{'='*65}")
print("PREDICCIONES SOBRE TEST")
print(f"{'='*65}")
print(f"Registros: {len(test_labels):,}")
print(f"\nDistribución predicha:")
pred_counts = pd.Series(test_labels).value_counts().reindex(["Low", "Medium", "High"])
for cls, cnt in pred_counts.items():
    print(f"  {cls:8s}: {cnt:>7,}  ({cnt/len(test_labels)*100:.2f}%)")
print(f"{'='*65}")

palette = {"Low": "steelblue", "Medium": "goldenrod", "High": "tomato"}
fig, ax = plt.subplots(figsize=(7, 4))
pred_counts.plot.bar(ax=ax, color=[palette[c] for c in pred_counts.index], edgecolor="black")
ax.set_title("Distribución de predicciones sobre Test", fontsize=13, fontweight="bold")
ax.set_ylabel("Cantidad"); ax.set_xlabel("Irrigation Need")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import os

submission = pd.DataFrame({"id": ids_test, TARGET: test_labels})
submission_path = "../data/4_catboost_submission.csv"
submission.to_csv(submission_path, index=False)

print(f"{'='*65}")
print("SUBMISSION GENERADA")
print(f"{'='*65}")
print(f"Archivo:  {os.path.abspath(submission_path)}")
print(f"Filas:    {len(submission):,}")
print(f"\nPrimeras filas:")
print(submission.head(5).to_string(index=False))
print(f"\n--- Resumen ---")
print(f"  CatBoost Base  — Accuracy: {np.mean(base_accs):.4f} ± {np.std(base_accs):.4f}")
print(f"  CatBoost Tuned — Accuracy: {np.mean(tuned_accs):.4f} ± {np.std(tuned_accs):.4f}")
print(f"  Modelo usado:              {best_label}")
print(f"{'='*65}")

## 7. Envío a Kaggle

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade kaggle --quiet

In [ ]:
COMPETITION = "playground-series-s6e4"
FILE        = "../data/4_catboost_submission.csv"
MSG         = f"CatBoost {best_label} - categóricas nativas - Acc={np.mean(best_accs):.4f}"

!kaggle competitions submit -c {COMPETITION} -f "{FILE}" -m "{MSG}"